# SQL Basics for Health Data Science

<!--- 
This notebook introduces SQL basics using real health data from NHANES. We'll cover:
- Basic SELECT queries
- Filtering and sorting
- Working with multiple tables
- Common health data analysis patterns
--->

## Setup and Imports
%pip install jupysql duckdb-engine pandas polars --quiet

In [1]:
from sqlalchemy import create_engine
import json
import pandas as pd

%load_ext sql

# Configure SQL magic for better output
%config SqlMagic.autopandas = True
%config SqlMagic.autocommit=True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Connect to DuckDB
# Execute SQL queries with pandas' `read_sql` function
# results_df = pd.read_sql("SELECT * FROM demographics LIMIT 5", engine)
engine = create_engine('duckdb:///:memory:')

# Use this engine for jupysql
%sql engine

# NOTE: memory db's are not shared between connections
# If using only jupysql, you can use the following:
# %sql duckdb:///:memory:

## Load Data

Let's load our NHANES data directly from CSV files:

In [2]:
%%sql
-- Clean up any existing tables
DROP TABLE IF EXISTS questionnaire;
DROP TABLE IF EXISTS laboratory;
DROP TABLE IF EXISTS examination;
DROP TABLE IF EXISTS demographics;

-- Load demographics
CREATE TABLE demographics AS
SELECT * FROM read_csv_auto('data/demographics.csv');

-- Load examination
CREATE TABLE examination AS
SELECT * FROM read_csv_auto('data/examination.csv');

-- Load laboratory
CREATE TABLE laboratory AS
SELECT * FROM read_csv_auto('data/labs.csv');

-- Load questionnaire
CREATE TABLE questionnaire AS
SELECT * FROM read_csv_auto('data/questionnaire.csv');

,Success


# Show the Data Dictionary

In [3]:
%%sql data_dictionary << 
-- Show data dictionary for all tables
SELECT 
    'demographics' as table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'demographics'
UNION ALL
SELECT 
    'examination' as table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'examination'
UNION ALL
SELECT 
    'laboratory' as table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'laboratory'
UNION ALL
SELECT 
    'questionnaire' as table_name,
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'questionnaire'
ORDER BY table_name, column_name;

In [4]:
display(data_dictionary)

,table_name,column_name,data_type
0,demographics,AIALANGA,BIGINT
1,demographics,DMDBORN4,BIGINT
2,demographics,DMDCITZN,BIGINT
3,demographics,DMDEDUC2,BIGINT
4,demographics,DMDEDUC3,BIGINT
...,...,...,...
1643,questionnaire,WHQ060,BIGINT
1644,questionnaire,WHQ070,BIGINT
1645,questionnaire,WHQ150,BIGINT
1646,questionnaire,WHQ500,BIGINT


In [5]:
# Or use the one I made for you, 'nhanes_data_dictionary.json'

# Load the data dictionary
with open('nhanes_data_dictionary.json', 'r') as f:
    data_dict = json.load(f)

# Convert to DataFrame
df_dict = []
for table, columns in data_dict.items():
    for col, desc in columns.items():
        df_dict.append({
            'Table': table,
            'Column': col,
            'Description': desc
        })

# Display as DataFrame
pd.DataFrame(df_dict)

,Table,Column,Description
0,demographics,SEQN,Respondent sequence number
1,demographics,RIDAGEYR,Age in years at screening
2,demographics,RIAGENDR,"Gender (1 = Male, 2 = Female)"
3,demographics,RIDRETH1,"Race/Hispanic origin (1 = Mexican American, 2 ..."
4,demographics,DMDEDUC2,Education level - Adults 20+ (1 = Less than 9t...
5,demographics,DMDMARTL,"Marital status (1 = Married, 2 = Widowed, 3 = ..."
6,demographics,INDFMPIR,Ratio of family income to poverty
7,examination,SEQN,Respondent sequence number
8,examination,BMXWT,Weight (kg)
9,examination,BMXHT,Standing height (cm)


## Basic SELECT Queries

Let's start with some simple SELECT queries:

In [6]:
%%sql
-- Get first 5 rows from demographics
SELECT * 
FROM demographics 
LIMIT 5;

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHREDU,DMDHRMAR,DMDHSEDU,WTINT2YR,WTMEC2YR,SDMVPSU,SDMVSTRA,INDHHIN2,INDFMIN2,INDFMPIR
0,73557,8,2,1,69,<NA>,4,4,1,<NA>,...,3,4,<NA>,13281.237386,13481.042095,1,112,4,4,0.84
1,73558,8,2,1,54,<NA>,3,3,1,<NA>,...,3,1,1,23682.057386,24471.769625,1,108,7,7,1.78
2,73559,8,2,1,72,<NA>,3,3,2,<NA>,...,4,1,3,57214.803319,57193.285376,1,109,10,10,4.51
3,73560,8,2,1,9,<NA>,3,3,1,119,...,3,1,4,55201.178592,55766.512438,2,109,9,9,2.52
4,73561,8,2,2,73,<NA>,3,3,1,<NA>,...,5,1,5,63709.667069,65541.871229,2,116,15,15,5.00


## Filtering Data

In [7]:
%%sql
-- Get participants over 60 years old
SELECT SEQN, RIDAGEYR, RIAGENDR
FROM demographics
WHERE RIDAGEYR > 60
ORDER BY RIDAGEYR DESC
LIMIT 10;

,SEQN,RIDAGEYR,RIAGENDR
0,73621,80,1
1,73628,80,1
2,73721,80,2
3,73722,80,2
4,73725,80,2
5,73732,80,1
6,73747,80,2
7,73774,80,2
8,73867,80,2
9,73873,80,1


## Joining Tables

In [8]:
%%sql
-- Combine demographics with blood pressure measurements
SELECT d.SEQN, d.RIDAGEYR, d.RIAGENDR,
       e.BPXSY1, e.BPXDI1
FROM demographics d
JOIN examination e ON d.SEQN = e.SEQN
WHERE d.RIDAGEYR > 60
LIMIT 10;

,SEQN,RIDAGEYR,RIAGENDR,BPXSY1,BPXDI1
0,73557,69,1,122,72
1,73559,72,1,140,90
2,73561,73,2,136,86
3,73564,61,2,118,80
4,73567,65,1,140,78
5,73571,76,1,124,68
6,73604,69,2,116,68
7,73607,75,1,120,84
8,73615,65,2,138,56
9,73616,62,2,126,50


## Aggregation and Grouping

In [9]:
%%sql
-- Average blood pressure by age group
SELECT 
    CASE 
        WHEN RIDAGEYR < 30 THEN '18-29'
        WHEN RIDAGEYR < 40 THEN '30-39'
        WHEN RIDAGEYR < 50 THEN '40-49'
        WHEN RIDAGEYR < 60 THEN '50-59'
        ELSE '60+'
    END as age_group,
    AVG(BPXSY1) as avg_systolic,
    AVG(BPXDI1) as avg_diastolic,
    COUNT(*) as n_participants
FROM demographics d
JOIN examination e ON d.SEQN = e.SEQN
GROUP BY age_group
ORDER BY age_group;

,age_group,avg_systolic,avg_diastolic,n_participants
0,18-29,107.966381,58.289537,5144
1,30-39,115.717149,70.763920,961
2,40-49,119.038793,73.329741,1007
3,50-59,125.229292,74.163265,916
4,60+,133.768461,67.843554,1785


## Practice

Try these exercises:
1. Find the average BMI by gender
2. Count the number of participants in each age group
3. Calculate the percentage of participants with high blood pressure
4. Find the correlation between age and blood pressure